In [ ]:
import pandas as pd
from IPython.display import display, clear_output, Audio
import ipywidgets as widgets
import json
import random
import requests
import base64
import threading
from google.oauth2 import service_account
from google.auth.transport.requests import Request

In [ ]:
mars_df = pd.read_csv('data/mars_dataset.csv')
mars_df

In [ ]:
#determine martian scout ranks and the xp needed to achieve each one

ranks = ["Tiger", "Wolf", "Bear", "Gryffin", "Arrow of Light"]
xp_needed = {"Tiger": 0, "Wolf": 50, "Bear": 150, "Gryffin": 300, "Arrow of Light": 500}

print(mars_df.unlock_rank.value_counts())

In [ ]:
# progress is saved in a dictionary so every function can read and change it

scout = {
    "rank": "Tiger",
    "xp": 0,
    "journal": [],      # names of creatures we've scanned
    "badges": [],       # merit badges we've earned
    "encounter": None,  # the creature in front of us right now
    "tries_left": 0
}

In [ ]:
# location is unlocked once your rank is high enough for at least one creature that lives there
# The legendary pull, Areonyx, lives at an unknown location so leave it out the map

def unlocked_locations():
    my_level = ranks.index(scout["rank"])
    allowed = mars_df[mars_df.unlock_rank.map(ranks.index) <= my_level]
    allowed = allowed[allowed.location_name != "Unknown (roams all regions)"]
    return sorted(allowed.location_name.unique())

In [ ]:
# prior to exploring, the scout checks the conditions of the environment including temp, radiation, dust storm risk, and ice

def location_report(location):
    place = mars_df[mars_df.location_name == location].iloc[0]
    print(f"📍 {location} ({place.region_type})")
    print(f"   Temperature: {place.avg_temp_c}°C   Radiation: {place.radiation_msv_per_sol} mSv/sol")
    print(f"   Dust storm risk: {int(place.dust_storm_risk_1to10)}/10   Ice: {place.ice_presence}")
    print()

In [ ]:
# a field card is the alien's attributes (inspired by pokemon), including type, rarity, danger level, and a scout tip

def show_field_card(creature):
    types = creature.primary_type
    if pd.notna(creature.secondary_type):
        types = types + " / " + creature.secondary_type
    print(f"  {creature['name']} - {creature.species_class}")
    print(f"  Type: {types}   Rarity: {creature.rarity}   Danger: {'⚠️' * creature.danger_level_1to5}")
    print(f"  {creature.description}")
    print(f"  🏕️ Scout tip: {creature.scout_tip}")
    print()

In [ ]:
# find aliens that common ones show up more often than rare ones

def find_creature(location):
    my_level = ranks.index(scout["rank"])

    # Areonyx, the legendary pull, has a small chance to show up anywhere once you reach Arrow of Light
    if scout["rank"] == "Arrow of Light" and random.random() < 0.03:
        return mars_df[mars_df['name'] == "Areonyx"].iloc[0]

    here = mars_df[(mars_df.location_name == location) & (mars_df.unlock_rank.map(ranks.index) <= my_level)]
    return here.sample(weights=here.catch_rate_pct).iloc[0]

In [ ]:
# add the XP, and move up as many ranks as the new total allows
# returns the new rank if the scout ranked up, otherwise none

def check_rank_up():
    new_rank = None
    for rank in ranks:
        if scout["xp"] >= xp_needed[rank] and ranks.index(rank) > ranks.index(scout["rank"]):
            scout["rank"] = rank
            new_rank = rank
            print(f"🎉 RANK UP! You reached the {rank} rank!")
            print(f"   New places to explore: {', '.join(unlocked_locations())}")
    return new_rank

In [ ]:
# every rank up gets its own song, made by Lyria (Google's music AI) from the name of the new rank
# if the key doesn't work, the game still runs, and the scout hears why at their next rank up

try:
    creds = service_account.Credentials.from_service_account_file(
        "keys/service.json",
        scopes=["https://www.googleapis.com/auth/cloud-platform"],
    )
    creds.refresh(Request())
    token = creds.token
    key_error = None
except Exception as e:
    token = None
    key_error = e

lyria_url = "https://aiplatform.googleapis.com/v1beta1/projects/wpp-cto-smartprod-sandbox/locations/global/interactions"

def rank_up_song(rank):
    if token is None:
        print(f"🔇 No {rank} song this time, the Google key didn't work ({key_error})")
        return None

    print(f"🎵 Composing your {rank} song...")
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": "lyria-3-clip-preview",
        "input": [
            {
                "type": "text",
                "text": f"A short, upbeat, triumphant celebration song for a young scout exploring Mars "
                        f"who just earned the {rank} rank. The chorus cheers '{rank}!'"
            }
        ]
    }

    # if anything goes wrong, skip the song so the game keeps going
    try:
        response = requests.post(lyria_url, headers=headers, json=payload, timeout=120)
        response.raise_for_status()
        # the song is the output that carries audio data
        songs = [o["data"] for o in response.json()["outputs"] if "data" in o]
        mp3_bytes = base64.b64decode(songs[-1])
    except Exception as e:
        print(f"🔇 Couldn't make the {rank} song this time ({e})")
        return None

    # autoplay plays it once as soon as it shows up
    return Audio(mp3_bytes, autoplay=True)

In [ ]:
def mars_expedition():

    # close the game panel from before, and the old songs in it, so every restart starts clean
    widgets.Widget.close_all()

    # UI Widgets
    title = widgets.HTML("<h2>🚀 Martian Scout Alien Expedition 👽</h2>"
                         "<b>1️⃣ Pick a place &nbsp; 2️⃣ Explore! &nbsp; 3️⃣ Scan (3 tries) &nbsp; 4️⃣ Earn XP to rank up</b>")
    location_picker = widgets.Dropdown(
        options=unlocked_locations(),
        description='Go to:',
        layout=widgets.Layout(width='350px')
    )
    explore_button = widgets.Button(description='Explore!', button_style='warning')
    scan_button = widgets.Button(description='Scan', button_style='success', disabled=True)
    journal_button = widgets.Button(description='Field Journal', button_style='info')
    status = widgets.HTML()
    output = widgets.Output()
    music = widgets.Output()  # its own area, so exploring again doesn't cut the song off

    # shows rank and XP at the top so the scout 
    def update_status():
        next_ranks = [r for r in ranks if xp_needed[r] > scout["xp"]]
        goal = f" (next: {next_ranks[0]} at {xp_needed[next_ranks[0]]} XP)" if next_ranks else ""
        status.value = (f"<b>Current Rank:</b> {scout['rank']} &nbsp; <b>XP:</b> {scout['xp']}{goal} &nbsp; "
                        f"<b>Journal:</b> {len(scout['journal'])}/{len(mars_df)}")

    # pause the game while a rank up song loads and plays, then let the scout keep going
    def lock_game():
        location_picker.disabled = True
        explore_button.disabled = True
        journal_button.disabled = True

    def unlock_game():
        location_picker.disabled = False
        explore_button.disabled = False
        journal_button.disabled = False

    def on_explore_click(b):
        with output:
            clear_output()
            location = location_picker.value
            location_report(location)

            # the higher the dust storm risk, the more likely a storm sends you back to the rover
            place = mars_df[mars_df.location_name == location].iloc[0]
            if random.random() < place.dust_storm_risk_1to10 * 0.05:
                print("🌪️ A dust storm rolls in! You head back to the rover. Try exploring again.")
                scan_button.disabled = True
                return

            creature = find_creature(location)
            scout["encounter"] = creature
            scout["tries_left"] = 3

            new = "" if creature['name'] in scout["journal"] else "  ✨ NEW!"
            print(f"👀 You spotted something... it's {creature['name']}!{new}")
            show_field_card(creature)
            print(f"Press Scan to add it to your journal. You have {scout['tries_left']} tries.")
            scan_button.disabled = False

    def on_scan_click(b):
        with output:
            creature = scout["encounter"]
            scout["tries_left"] -= 1

            # catch_rate_pct is the chance (out of 100) that one scan works
            if random.random() * 100 < creature.catch_rate_pct:
                print(f"📡 Scan complete! {creature['name']} was added to your Field Journal.")
                scout["xp"] += creature.scan_xp
                print(f"   +{creature.scan_xp} XP")

                if creature['name'] not in scout["journal"]:
                    scout["journal"].append(creature['name'])
                if creature.merit_badge not in scout["badges"]:
                    scout["badges"].append(creature.merit_badge)
                    print(f"🏅 You earned the {creature.merit_badge} merit badge!")

                new_rank = check_rank_up()
                location_picker.options = unlocked_locations()
                scan_button.disabled = True
                update_status()

                # play one song for the new rank, and wait for it to finish before the game continues
                if new_rank:
                    lock_game()
                    song = rank_up_song(new_rank)
                    if song:
                        with music:
                            clear_output()
                            display(song)
                        # Lyria songs are 192 kbps mp3s, so the file size tells us how long the song is
                        song_seconds = len(song.data) * 8 / 192000
                        print(f"🎶 Enjoy your {new_rank} song! The game continues when it ends.")

                        def song_over():
                            unlock_game()
                            output.append_stdout("▶️ Song's over, keep exploring!\n")

                        threading.Timer(song_seconds + 1, song_over).start()
                    else:
                        unlock_game()
            elif scout["tries_left"] > 0:
                print(f"...the scan was fuzzy. {scout['tries_left']} tries left.")
            else:
                print(f"💨 {creature['name']} got away! Pick a place and explore again.")
                scan_button.disabled = True

            update_status()

    def on_journal_click(b):
        with output:
            clear_output()
            if not scout["journal"]:
                print("Your Field Journal is empty. Go explore!")
                return
            journal = mars_df[mars_df['name'].isin(scout["journal"])]
            display(journal[["dex_no", "name", "primary_type", "rarity", "location_name", "merit_badge"]])
            print(f"🏅 Badges: {', '.join(scout['badges'])}")

    # connect functions to the buttons
    explore_button.on_click(on_explore_click)
    scan_button.on_click(on_scan_click)
    journal_button.on_click(on_journal_click)

    # display the widgets
    update_status()
    display(widgets.VBox([title, status, location_picker, widgets.HBox([explore_button, scan_button, journal_button]), music, output]))

In [ ]:
mars_expedition()